In [1]:
# Select the mode and configure feature and model-data paths.

import os

mode = "replay"
os.environ["COMMENTGAP_MODE"] = mode
os.environ['COMMENTGAP_FROZEN_REPLAY'] = '1' if mode == 'recompute' else '0'

from pathlib import Path
import pandas as pd
from IPython.display import display
import json
from commentgap_analysis.public_replay import verify_public_stage
import pyarrow.parquet as pq
if mode != 'replay':
    from commentgap_analysis.preprocessing import prepare_shared_model_data
    from commentgap_analysis.paths import ExecutionContext, ProjectPaths
    from commentgap_analysis.frozen_inputs import unavailable
    from commentgap_analysis.notebook_paths import configure_notebook_stage

if mode == 'replay':
    PUBLIC_ROOT = Path(os.environ.get('COMMENTGAP_PUBLIC_ARTIFACT_ROOT', Path.cwd() / 'artifacts/canonical')).resolve()
    verify_public_stage('03', root=PUBLIC_ROOT)
elif mode == 'recompute':
    _cg_context = ExecutionContext.from_values(mode="frozen")
    os.environ['COMMENTGAP_FEATURE_ROOT'] = str(_cg_context.read_root('features'))
    os.environ['COMMENTGAP_MODEL_DATA_ROOT'] = str(_cg_context.read_root('model_data'))
elif mode in ('fresh', 'resume'):
    configure_notebook_stage('03')


# 3. Shared Paper 1 split and model preprocessing

Preserve the existing 50/50 article split and create one common feature contract for all Paper 1 ranking models. Shared transformations produce author reception rates, discussion pace, prior reply composition, and leakage-safe centred reply depth. Recent-activity share and branch activity counts are excluded from the primary contract. The source path retains a legacy held-out-analysis name until the post-factorial compatibility migration.

### Execution modes for this notebook

`replay` (default) verifies and displays the compact public aggregate products from `artifacts/canonical/`. `recompute` executes the saved-product cells using private `data/` and `artifacts/archive/recompute/`. `fresh` runs this stage’s producer from retained inputs; see [reproduction modes](docs/reproduction.md) for prerequisites. `resume` continues a recorded producer run.


In [ ]:
# Set preprocessing options and check that the frozen split is available.

if mode != 'replay':
    SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
    FEATURE_ROOT = Path(os.environ["COMMENTGAP_FEATURE_ROOT"])
    MODEL_DATA_ROOT = Path(os.environ["COMMENTGAP_MODEL_DATA_ROOT"])
    LEGACY_SPLIT = Path(os.getenv("COMMENTGAP_FROZEN_SPLIT", str(ProjectPaths.load().configured("source_split"))))
    FORCE = os.getenv("COMMENTGAP_PREPROCESS_FORCE", "0").lower() in {"1", "true", "yes"}
    assert LEGACY_SPLIT.exists(), f"Frozen legacy split is missing: {LEGACY_SPLIT}"
    {"feature_root": str(FEATURE_ROOT), "output_root": str(MODEL_DATA_ROOT), "frozen_split": str(LEGACY_SPLIT), "force": FORCE}


In [ ]:
# Load the public model-data inventory or prepare the shared model tables.

if mode == 'replay':
    model_inventory = json.loads((PUBLIC_ROOT / 'shared/model_data/model_inventory_public.json').read_text())
    model_columns = model_inventory['column_names']
    model_contract = pd.DataFrame([{'rows': model_inventory['row_count'], 'columns': len(model_columns),
        'aqua expected features': sum(n.startswith('aqua_') and n.endswith('_expected') for n in model_columns)}])
elif mode == 'recompute':
    # Recompute restores the prepared split and model tables from saved products.
    required = [MODEL_DATA_ROOT / name for name in (
        "master_article_split.parquet", "split_balance_diagnostics.csv",
        "preprocessing_parameters.json", "feature_manifest.json",
        "choice_set_root.parquet", "choice_set_all.parquet",
    )]
    missing = [path for path in required if not path.is_file()]
    if missing:
        unavailable("result", missing[0], "The saved shared preprocessing result cannot be restored; downstream result displays are skipped.")
        _result_available = False
    else:
        _result_available = True
        parameters = json.loads((MODEL_DATA_ROOT / "preprocessing_parameters.json").read_text())
        feature_manifest = json.loads((MODEL_DATA_ROOT / "feature_manifest.json").read_text())
        result = {
            "article_split": pd.read_parquet(MODEL_DATA_ROOT / "master_article_split.parquet"),
            "split_balance": pd.read_csv(MODEL_DATA_ROOT / "split_balance_diagnostics.csv"),
            "reply_depth_centers": parameters["reply_depth_centers"],
            "feature_manifest": feature_manifest,
            "output_root": str(MODEL_DATA_ROOT),
            "rows": {scope: pq.ParquetFile(MODEL_DATA_ROOT / f"choice_set_{scope}.parquet").metadata.num_rows for scope in ("root", "all")},
            "cache_hit": True,
        }
    if _result_available:
        result["rows"]
elif mode in ('fresh', 'resume'):
    # Fresh runs construct the shared data from feature products.
    result = prepare_shared_model_data(
        FEATURE_ROOT, MODEL_DATA_ROOT, frozen_split_path=LEGACY_SPLIT,
        seed=SEED, development_folds=5, force=FORCE,
    )
    result["rows"]


In [4]:
# Review split balance, model columns, and transformed predictor coverage.

if mode == 'replay':
    split_role_counts = pd.read_csv(PUBLIC_ROOT / 'shared/model_data/split_role_counts.csv')
    split_balance = pd.read_csv(PUBLIC_ROOT / 'shared/model_data/split_balance_diagnostics.csv')
    reply_depth_centers = json.loads((PUBLIC_ROOT / 'shared/model_data/reply_depth_centers.json').read_text())
    can_display = True
else:
    can_display = not (mode == 'recompute' and not _result_available)
    if can_display:
        model_schema = pq.ParquetFile(MODEL_DATA_ROOT / 'choice_set_all.parquet')
        model_columns = model_schema.schema_arrow.names
        model_inventory = {'row_count': model_schema.metadata.num_rows, 'column_names': model_columns}
        split_role_counts = result['article_split']['split_role'].value_counts().rename_axis('split_role').reset_index(name='articles')
        split_balance = result['split_balance']
        reply_depth_centers = result['reply_depth_centers']
    else:
        print('Skipped result displays because the saved preprocessing inputs are unavailable.')

# Prepare a common set of tables for the mode-independent displays.
if can_display:
    model_contract = pd.DataFrame([{'rows': model_inventory['row_count'], 'columns': len(model_columns),
        'aqua expected features': sum(n.startswith('aqua_') and n.endswith('_expected') for n in model_columns)}])
    display(model_contract)
    display(pd.DataFrame({'column': model_columns}), split_role_counts, split_balance,
            pd.DataFrame([{'group': k, 'center': v} for k, v in reply_depth_centers.items()]))

    # Inspect transformed predictors when the full feature manifest is available.
    if mode != 'replay':
        for scope in ('root', 'all'):
            features = result['feature_manifest']['models'][scope]['features']
            transformed = {'prior_reply_composition', 'discussion_pace', 'reply_depth_centered',
                'author_prior_30d_upvote_reception', 'author_prior_30d_downvote_reception'}
            print(scope, len(features), [name for name in features if name in transformed])


,rows,columns,aqua expected features
0,2392952,116,21


,column
0,story_id
1,comment_id
2,article_year
3,article_month
4,candidate_scope
...,...
111,reply_depth_centered_fold_00
112,reply_depth_centered_fold_01
113,reply_depth_centered_fold_02
114,reply_depth_centered_fold_03


,split_role,articles
0,development,2559
1,paper2_test,2559


,covariate,development_mean,paper2_test_mean,standardized_mean_difference,abs_standardized_mean_difference,acceptance_threshold,accepted
0,log1p_n_candidates_root,4.573854,4.572367,0.001707,0.001707,0.05,True
1,log1p_n_candidates_all,5.737229,5.757111,-0.021059,0.021059,0.05,True
2,n_picks_root,3.940211,3.932395,0.004736,0.004736,0.05,True
3,n_picks_all,4.091833,4.087143,0.002777,0.002777,0.05,True
4,reply_proportion,0.669957,0.675579,-0.046128,0.046128,0.05,True


,group,center
0,fold_00_training,1.096526
1,fold_01_training,1.099113
2,fold_02_training,1.097237
3,fold_03_training,1.095427
4,fold_04_training,1.099265
5,full_development,1.097516


## Downstream contract

All regression, XGBoost, and neural ranking workflows read the same v4 model-data contract. It contains discussion pace instead of total prior-comment counts, omits recent-activity share and branch activity, and substitutes fold-specific centred reply depth during development CV.